# DataPilot AI Pro — Real Mistral-7B Fine-Tuning (QLoRA / DoRA / DPO)

Trains the **real production target** (`mistralai/Mistral-7B-Instruct-v0.3`)
that a local 4 GB laptop GPU could not run — this notebook needs a Colab GPU
runtime (T4 free tier, or L4/A100 on Colab Pro).

**What this notebook reuses vs. builds new:**
- **Reused, unchanged**: the training dataset (756 examples: 606 train / 75
  val / 75 test + DPO preference pairs), already built and validated by
  `finetuning/dataset_prep.py` — pulled from the GitHub repo below, not
  regenerated.
- **Reused, adapted**: the exact training recipes from `finetune_qlora.py`,
  `finetune_dora.py`, `finetune_dpo.py`, `benchmark.py`, and
  `export_ollama.py` — inlined here (not `git clone`'d) because several
  local bug fixes to those scripts are **uncommitted** as of this writing;
  inlining guarantees this notebook runs the validated logic regardless of
  push status. Hyperparameters are the scripts' own non-smoke defaults
  (this is the first run with enough VRAM to use them un-reduced).
- **New in this notebook**: real quantization via a compiled `llama-quantize`
  (the earlier local export only produced f16 GGUF because no C++ toolchain
  was available there — this notebook builds it for real).
- **Explicitly out of scope** (per instruction): ORPO and GaLore are not run
  here.

**Honesty note**: every "print real numbers" cell below computes those
numbers live from `torch.cuda`/`trainer.state.log_history` — nothing is
pre-filled. Outputs are empty until you run each cell in Colab.

**Runtime checklist before you start:**
1. `Runtime > Change runtime type` → **T4 GPU** (or L4/A100 on Pro).
2. `Runtime > Run all`, or run cells top-to-bottom — each stage prints a
   checkpoint before the next one starts.
3. Keep the browser tab open (Colab free tier disconnects idle sessions).


## Time & VRAM budget (planning estimates — NOT measurements)

These are rough planning ranges from published QLoRA/DoRA/Unsloth
benchmarks at this model size, **not** numbers from a run of this notebook.
The real numbers this run actually produces are printed by each stage below.

| Stage | Rough time (T4) | Peak VRAM (rough) |
|---|---|---|
| Install + GPU check | 3-5 min | — |
| Dataset load | <1 min | — |
| QLoRA (3 epochs, 606 ex) | 20-45 min | ~7-9 GB |
| DoRA (3 epochs, 606 ex, no Unsloth speedup) | 30-60 min | ~8-10 GB |
| DPO (1 epoch, 606 pairs, on QLoRA adapter) | 20-45 min | ~8-10 GB |
| Build llama.cpp (`llama-quantize` only, no CUDA) | 3-8 min | — |
| Merge + convert + quantize (q4_k_m) | 8-15 min | — |
| 4-way benchmark (generation + Ollama judge) | 20-40 min | varies |
| **Total** | **~2-3.5 hours** | fits T4's 16 GB throughout |

**If time/memory looks tight partway through DoRA**: DoRA is independent of
DPO (DPO only needs the QLoRA adapter, not DoRA's). You can skip the DoRA
cells entirely and continue straight to the DPO section — flagged again at
that cell. Each training stage also backs its adapter up to Google Drive
immediately after saving, so a session drop after any stage does not lose
that stage's work.


## Step 0 — Confirm a usable GPU is attached (fails loudly if not)

In [ ]:
# Uses Colab's pre-installed torch — checked BEFORE any pip installs so a
# missing/wrong GPU fails immediately instead of after a 5-minute install.
import torch, subprocess

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout
      or "nvidia-smi returned nothing")

if not torch.cuda.is_available():
    raise RuntimeError(
        "\n\n*** NO GPU DETECTED ***\n"
        "Go to Runtime > Change runtime type > Hardware accelerator > GPU "
        "(T4 on the free tier), then Runtime > Restart session, then re-run "
        "this cell. Do not proceed without a GPU — Mistral-7B QLoRA needs "
        "~7 GB VRAM minimum; there is no CPU fallback in this notebook."
    )

gpu_name = torch.cuda.get_device_name(0)
total_vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU: {gpu_name}")
print(f"Total VRAM: {total_vram_gb:.1f} GB")

# Gate on VRAM, not a strict name whitelist: a P100/V100-16GB/A100 all work
# fine for this recipe even though the user asked for "T4/L4" specifically —
# the real requirement is enough memory, not the exact model name.
MIN_VRAM_GB = 14.0
if total_vram_gb < MIN_VRAM_GB:
    raise RuntimeError(
        f"\n\n*** GPU TOO SMALL ***\n"
        f"Detected {gpu_name} with {total_vram_gb:.1f} GB VRAM; this recipe "
        f"needs at least {MIN_VRAM_GB} GB (T4=16GB / L4=24GB / A100=40GB+). "
        f"On Colab free tier you may have been assigned a smaller/shared "
        f"GPU — try Runtime > Disconnect and delete runtime, then "
        f"reconnect, or upgrade to Colab Pro for guaranteed T4/L4/A100."
    )

# T4 (Turing, compute capability 7.5) has NO native bf16 support; L4 (Ada,
# 8.9) and A100 (Ampere, 8.0) do. Using bf16 on unsupported hardware either
# silently falls back to a slow path or errors depending on transformers
# version — so every training cell below reads this flag instead of
# hardcoding bf16=True the way the original desktop scripts did (the dev
# machine's RTX A500 is Ampere and never needed this branch).
USE_BF16 = torch.cuda.is_bf16_supported()
print(f"bf16 supported on this GPU: {USE_BF16} "
      f"({'will train in bf16' if USE_BF16 else 'will train in fp16 instead'})")
print("\nGPU check passed — safe to proceed.")


## Step 1 — Install dependencies

Pins match `finetuning/requirements-finetuning.txt` (the versions already
validated locally), installed as an explicit override AFTER Unsloth's own
installer so Unsloth's dependency resolution doesn't silently drift them.

One deviation from the local setup: the local Windows machine had to pin
`pyarrow==16.1.0` to work around a Windows-only `torch2.6 + pyarrow24 +
bitsandbytes` access-violation crash. That crash is specific to Windows DLL
loading order and does not occur on Colab's Linux environment, so it is
intentionally **not** applied here.


In [ ]:
%%capture install_log
!pip install --upgrade -q pip
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
# NOTE: an earlier version of this cell force-pinned trl==0.9.6/peft==0.12.0/
# accelerate==0.33.0 with --no-deps, matching what was validated locally.
# That broke on a real run: Unsloth's installer resolves a modern,
# mutually-compatible stack for whatever torch/CUDA the runtime has
# (observed on an L4: transformers 5.5.0, torch 2.11.0+cu128) -- far newer
# than existed when this notebook was built -- and the old --no-deps
# accelerate pin is incompatible with it ("Using a device_map... requires
# accelerate" even though accelerate IS installed, just too old for
# transformers to recognize as usable). Trusting Unsloth's own dependency
# resolution now instead of fighting it; the trainer-construction cells
# below use introspection to adapt to whichever trl API shape results,
# rather than assuming one fixed version.
!pip install -q -U bitsandbytes accelerate
!pip install -q sentencepiece protobuf


In [ ]:
# Show the tail of the install log (full log captured above in case you
# need to scroll back for a specific error).
print("\n".join(install_log.stdout.splitlines()[-40:]))


In [ ]:
# Re-verify the GPU is still visible to torch after installs, confirm
# Unsloth imports cleanly and reports the same GPU, and print final versions
# actually installed (not assumed).
import torch, trl, peft, accelerate, datasets, transformers
import unsloth
from unsloth import FastLanguageModel

assert torch.cuda.is_available(), "GPU disappeared after install — restart runtime and re-run from Step 0."
print(f"torch          {torch.__version__}  (cuda available: {torch.cuda.is_available()})")
print(f"unsloth        {unsloth.__version__}")
print(f"trl            {trl.__version__}")
print(f"peft           {peft.__version__}")
print(f"accelerate     {accelerate.__version__}")
print(f"datasets       {datasets.__version__}")
print(f"transformers   {transformers.__version__}")
print(f"GPU (post-install): {torch.cuda.get_device_name(0)}")

# Fail fast HERE (seconds) instead of ~20 minutes into a training cell if
# transformers can't actually use the installed accelerate — this is
# exactly the check that silently failed deep inside
# FastLanguageModel.from_pretrained() previously.
from transformers.utils import is_accelerate_available
if not is_accelerate_available():
    raise RuntimeError(
        "\n\n*** transformers cannot detect a usable accelerate ***\n"
        "A version mismatch between transformers and accelerate. Try:\n"
        "  !pip install -q -U accelerate\n"
        "then Runtime > Restart session and re-run from Step 1 (installs "
        "persist across a session restart within the same runtime)."
    )
print("accelerate is usable by transformers: True")


## Step 2 — Mount Google Drive (recommended, not required)

Every adapter is copied to Drive immediately after training, so a dropped
session after any stage still leaves that stage's work recoverable. If you
decline the mount, training still proceeds — you'll only have the final
`files.download()` cells near the end as a fallback.


In [ ]:
import os

DRIVE_BACKUP_DIR = None
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_BACKUP_DIR = "/content/drive/MyDrive/datapilot_mistral7b_finetuning"
    os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)
    print(f"Drive mounted. Backups will go to: {DRIVE_BACKUP_DIR}")
except Exception as e:
    print(f"Drive mount skipped/failed ({e}). Continuing without Drive "
          f"backups — use the files.download() cells near the end instead.")

def backup_to_drive(local_dir: str, name: str):
    """Copy a trained adapter directory to Drive right after it's saved.
    No-op (prints a note) if Drive wasn't mounted."""
    if DRIVE_BACKUP_DIR is None:
        print(f"  (Drive not mounted — {name} only saved locally at {local_dir})")
        return
    import shutil
    dest = os.path.join(DRIVE_BACKUP_DIR, name)
    shutil.copytree(local_dir, dest, dirs_exist_ok=True)
    print(f"  Backed up to Drive: {dest}")


## Step 3 — Load the already-validated dataset (756 examples)

Reuses the exact output of `finetuning/dataset_prep.py` from a prior run —
**not regenerated here**. Three options, in order:

**Option A (try first, next cell)** — pulls the 5 `.jsonl` files directly
from the `fine-tune` branch via `raw.githubusercontent.com`. Works only if
the GitHub repo is **public**. `raw.githubusercontent.com` (and the public
GitHub API) return `HTTP 404` for a private repo when no token is given —
not 401/403 — that's deliberate on GitHub's side, so it doesn't leak
whether a private repo exists to anonymous requests. If you see 404s below,
your repo is private; that's not a bug, use Option A2 or B instead.

**Option A2 (if Option A 404'd)** — same download, authenticated with a
GitHub token, so it works against a private repo without changing its
visibility. Token is entered hidden via `getpass` and never written into
this notebook.

**Option B (always works, no token)** — manual upload. Zip
`finetuning/data/{train,val,test,preferences_train,preferences_val}.jsonl`
from your machine and upload it when prompted.


In [ ]:
import os, json, urllib.request, urllib.error

GITHUB_USER = "Shiva250503ss"          # change if your fork differs
GITHUB_REPO = "DataScienceTeamProject"
GITHUB_BRANCH = "fine-tune"            # branch confirmed to have finetuning/data/ pushed

DATA_FILES = ["train.jsonl", "val.jsonl", "test.jsonl",
              "preferences_train.jsonl", "preferences_val.jsonl"]
DATA_DIR = "/content/finetuning/data"
os.makedirs(DATA_DIR, exist_ok=True)

EXPECTED_COUNTS = {"train.jsonl": 606, "val.jsonl": 75, "test.jsonl": 75,
                   "preferences_train.jsonl": 606, "preferences_val.jsonl": 75}

RAW_BASE = f"https://raw.githubusercontent.com/{GITHUB_USER}/{GITHUB_REPO}/{GITHUB_BRANCH}/finetuning/data"

download_ok = True
saw_404 = False
for fname in DATA_FILES:
    url = f"{RAW_BASE}/{fname}"
    dest = os.path.join(DATA_DIR, fname)
    try:
        urllib.request.urlretrieve(url, dest)
        n_lines = sum(1 for _ in open(dest, encoding="utf-8"))
        print(f"  {fname}: downloaded, {n_lines} lines")
    except urllib.error.HTTPError as e:
        print(f"  {fname}: FAILED (HTTP {e.code})")
        download_ok = False
        saw_404 = saw_404 or (e.code == 404)
    except Exception as e:
        print(f"  {fname}: FAILED ({type(e).__name__}: {e})")
        download_ok = False

if download_ok:
    for fname, expected in EXPECTED_COUNTS.items():
        actual = sum(1 for _ in open(os.path.join(DATA_DIR, fname), encoding="utf-8"))
        status = "OK" if actual == expected else "MISMATCH"
        print(f"  [{status}] {fname}: expected {expected}, got {actual}")
    print("\nOption A succeeded -- dataset loaded from GitHub.")
elif saw_404:
    print(
        "\n*** HTTP 404 on one or more files -- this means the repo is PRIVATE. ***\n"
        "raw.githubusercontent.com returns 404 (not 401/403) for private repos\n"
        "when no token is supplied -- expected behavior, not a bug in this cell.\n\n"
        "Next step: run the 'Option A2' cell right below (token-based download,\n"
        "repo stays private), or skip to 'Option B' further down (manual upload,\n"
        "no token needed at all)."
    )
else:
    print("\nOption A failed for at least one file (non-404 error, see above) "
          "-- try Option A2 or Option B below.")


### Option A2 — authenticated download (only needed if Option A 404'd)

Same 5 files, fetched via the GitHub Contents API with a token so it works
against a private repo. The token is entered with `getpass` (input hidden,
never echoed, never written into this notebook's saved source or output)
and used only in-memory for these requests, then discarded.

**Create a token** (~2 min): GitHub -> Settings -> Developer settings ->
Personal access tokens -> Fine-grained tokens -> New token -> scope it to
this one repo -> Repository permissions -> Contents -> **Read-only** ->
Generate. Paste it when the cell below prompts for it.

Skip this cell if Option A already succeeded, or if you'd rather just use
Option B (manual upload, no token at all).


In [ ]:
RUN_AUTH_DOWNLOAD = False  # flip to True to actually run this cell's body

if RUN_AUTH_DOWNLOAD:
    import getpass, requests

    token = getpass.getpass("GitHub token (hidden input, not saved anywhere): ")
    headers = {"Authorization": f"token {token}",
              "Accept": "application/vnd.github.v3.raw"}
    api_base = f"https://api.github.com/repos/{GITHUB_USER}/{GITHUB_REPO}/contents/finetuning/data"

    download_ok = True
    for fname in DATA_FILES:
        r = requests.get(f"{api_base}/{fname}", headers=headers,
                         params={"ref": GITHUB_BRANCH}, timeout=30)
        dest = os.path.join(DATA_DIR, fname)
        if r.status_code == 200:
            with open(dest, "wb") as f:
                f.write(r.content)
            n_lines = sum(1 for _ in open(dest, encoding="utf-8"))
            print(f"  {fname}: downloaded (authenticated), {n_lines} lines")
        else:
            print(f"  {fname}: FAILED (HTTP {r.status_code}) -- check the "
                  f"token's scope/expiry and that GITHUB_USER/GITHUB_REPO/"
                  f"GITHUB_BRANCH above match your repo")
            download_ok = False
    del token  # drop it from memory now that we're done with it

    if download_ok:
        for fname, expected in EXPECTED_COUNTS.items():
            actual = sum(1 for _ in open(os.path.join(DATA_DIR, fname), encoding="utf-8"))
            status = "OK" if actual == expected else "MISMATCH"
            print(f"  [{status}] {fname}: expected {expected}, got {actual}")
        print("\nOption A2 succeeded -- dataset loaded from GitHub (authenticated).")
else:
    print("Skipped (RUN_AUTH_DOWNLOAD is False). Set it to True above if "
          "Option A 404'd and you'd rather use a token than upload manually.")


### Option B — manual upload (if Options A and A2 above didn't work, or you just prefer this)

On your local machine, zip `finetuning/data/train.jsonl`, `val.jsonl`,
`test.jsonl`, `preferences_train.jsonl`, `preferences_val.jsonl` together,
run the cell below, and select that zip when prompted.


In [ ]:
RUN_MANUAL_UPLOAD = False  # flip to True to actually run this cell's body

if RUN_MANUAL_UPLOAD:
    import zipfile
    from google.colab import files
    print("Select the zip containing the 5 finetuning/data/*.jsonl files...")
    uploaded = files.upload()
    zip_name = next(iter(uploaded))
    with zipfile.ZipFile(zip_name) as zf:
        zf.extractall(DATA_DIR)
    print(f"Extracted into {DATA_DIR}:")
    for fname in DATA_FILES:
        p = os.path.join(DATA_DIR, fname)
        print(f"  {fname}: {'found' if os.path.exists(p) else 'MISSING'}")
else:
    print("Skipped (RUN_MANUAL_UPLOAD is False). Set it to True above if "
          "Option A failed and you need to upload manually.")


In [ ]:
# ── Shared prompt template + data loaders ──────────────────────────────────
# Reused verbatim from finetuning/common.py — the SAME format used to build
# the dataset, so training text here is byte-identical in structure to what
# dataset_prep.py produced.

MAX_SEQ_LENGTH = 2048
SEED = 42

SYSTEM_INSTRUCTION = (
    "You are the Explainer agent of an AutoML platform. You receive the raw "
    "feature-attribution output of a machine-learning model (SHAP or LIME "
    "values) and must explain the prediction to a non-technical business "
    "user. Write 2-4 short sentences in plain English. Never use the words "
    "'SHAP', 'LIME', or any statistics jargon. Say WHY the prediction was "
    "made, WHICH factors mattered most, and WHAT would change the outcome."
)

def format_prompt(shap_input: str) -> str:
    return f"[INST] {SYSTEM_INSTRUCTION}\n\n{shap_input} [/INST]"

def format_example(shap_input: str, explanation: str, eos_token: str = "</s>") -> str:
    return f"{format_prompt(shap_input)} {explanation}{eos_token}"

def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

# Fail loudly and specifically if the data isn't actually there, instead of
# a bare FileNotFoundError three calls deep into load_jsonl(). Means none of
# Option A / A2 / B above has successfully populated DATA_DIR yet.
_missing = [f for f in DATA_FILES if not os.path.exists(os.path.join(DATA_DIR, f))]
if _missing:
    raise FileNotFoundError(
        f"\n\n*** DATASET NOT LOADED YET ***\n"
        f"Missing from {DATA_DIR}: {_missing}\n\n"
        f"Scroll up to Step 3:\n"
        f"  - If Option A printed HTTP 404, the repo is private -- run "
        f"Option A2 (token) or Option B (manual upload) instead.\n"
        f"  - If you already ran Option B, confirm the zip had these 5 "
        f"filenames directly at its top level, not inside a subfolder."
    )

train_rows = load_jsonl(os.path.join(DATA_DIR, "train.jsonl"))
val_rows = load_jsonl(os.path.join(DATA_DIR, "val.jsonl"))
test_rows = load_jsonl(os.path.join(DATA_DIR, "test.jsonl"))
pref_train_rows = load_jsonl(os.path.join(DATA_DIR, "preferences_train.jsonl"))
pref_val_rows = load_jsonl(os.path.join(DATA_DIR, "preferences_val.jsonl"))

print(f"SFT splits  — train: {len(train_rows)} | val: {len(val_rows)} | test: {len(test_rows)}")
print(f"DPO pairs   — train: {len(pref_train_rows)} | val: {len(pref_val_rows)}")
print("\n--- Example training text (formatted) ---")
print(format_example(train_rows[0]["input"], train_rows[0]["output"])[:500], "...")


## Step 4 — Shared measurement utility

Inlined from `finetuning/common.py`'s `RunTracker` — measures wall-clock
time and peak GPU memory identically for every stage below, so the three
runs (QLoRA / DoRA / DPO) are directly comparable, exactly as they are in
the local benchmark harness.


In [ ]:
import time

class RunTracker:
    def __init__(self, technique: str, output_dir: str):
        self.technique = technique
        self.output_dir = output_dir
        self.stats = {"technique": technique}
        self._t0 = None

    def start(self):
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
            self.stats["gpu"] = torch.cuda.get_device_name(0)
        self._t0 = time.time()

    def stop(self, final_loss=None, eval_loss=None, extra=None):
        self.stats["train_time_seconds"] = round(time.time() - self._t0, 1)
        self.stats["train_time_human"] = time.strftime(
            "%Hh %Mm %Ss", time.gmtime(self.stats["train_time_seconds"]))
        if torch.cuda.is_available():
            self.stats["peak_gpu_memory_gb"] = round(
                torch.cuda.max_memory_allocated() / 1024 ** 3, 2)
        if final_loss is not None:
            self.stats["final_train_loss"] = round(float(final_loss), 4)
        if eval_loss is not None:
            self.stats["final_eval_loss"] = round(float(eval_loss), 4)
        if extra:
            self.stats.update(extra)

    def save(self):
        os.makedirs(self.output_dir, exist_ok=True)
        path = os.path.join(self.output_dir, "training_stats.json")
        with open(path, "w", encoding="utf-8") as f:
            json.dump(self.stats, f, indent=2)
        print(f"\n=== [{self.technique}] REAL MEASURED RESULTS ===")
        for k, v in self.stats.items():
            print(f"  {k}: {v}")
        print(f"  (saved to {path})")

def get_final_losses(trainer):
    train_loss, eval_loss = None, None
    for entry in trainer.state.log_history:
        if "loss" in entry:
            train_loss = entry["loss"]
        if "eval_loss" in entry:
            eval_loss = entry["eval_loss"]
    return train_loss, eval_loss

def free_gpu(*objs):
    """Delete model/trainer objects and release VRAM before the next stage
    loads a fresh 7B copy — critical for staying inside a T4's 16 GB across
    three sequential training runs in one session."""
    import gc
    for o in objs:
        try:
            del o
        except NameError:
            pass
    gc.collect()
    torch.cuda.empty_cache()
    if torch.cuda.is_available():
        print(f"  GPU memory after cleanup: "
              f"{torch.cuda.memory_allocated()/1024**3:.2f} GB allocated, "
              f"{torch.cuda.memory_reserved()/1024**3:.2f} GB reserved "
              f"(peak this stage: {torch.cuda.max_memory_allocated()/1024**3:.2f} GB)")

BASE_MODEL_HF = "mistralai/Mistral-7B-Instruct-v0.3"
BASE_MODEL_UNSLOTH_4BIT = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit"
LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj",
                "gate_proj", "up_proj", "down_proj"]
MODELS_DIR = "/content/finetuning/models"
os.makedirs(MODELS_DIR, exist_ok=True)
print("Measurement utilities ready.")


# ── Adaptive trainer construction ────────────────────────────────────────
# trl's SFTTrainer/DPOTrainer constructor API has shifted across versions
# in ways this notebook can't safely pin against — Unsloth's installer
# resolves whatever transformers/trl/peft/accelerate stack is mutually
# compatible with the Colab runtime at the moment you run this, which can
# be far newer than any single version validated locally. These helpers
# use inspect.signature() to detect the installed API shape instead of
# hardcoding a guess, covering two documented trl API drifts:
#   1. `tokenizer=` was renamed to `processing_class=` in newer trl.
#   2. `dataset_text_field=`/`max_seq_length=` moved off SFTTrainer's own
#      constructor onto SFTConfig (a TrainingArguments subclass) in newer trl.
import inspect

def _tok_kwarg(trainer_cls) -> str:
    sig = inspect.signature(trainer_cls.__init__)
    return "processing_class" if "processing_class" in sig.parameters else "tokenizer"

def _build_sft_trainer(model, tok, train_ds, val_ds, output_dir, *,
                       num_train_epochs, per_device_train_batch_size,
                       gradient_accumulation_steps, learning_rate, optim,
                       run_name, warmup_ratio=0.03, gradient_checkpointing=False):
    from transformers import TrainingArguments
    from trl import SFTTrainer
    try:
        from trl import SFTConfig
    except ImportError:
        SFTConfig = None

    sft_sig = inspect.signature(SFTTrainer.__init__)
    accepts_directly = ("dataset_text_field" in sft_sig.parameters
                        and "max_seq_length" in sft_sig.parameters)

    common_targs = dict(
        output_dir=output_dir,
        num_train_epochs=num_train_epochs,
        per_device_train_batch_size=per_device_train_batch_size,
        gradient_accumulation_steps=gradient_accumulation_steps,
        learning_rate=learning_rate,
        lr_scheduler_type="cosine",
        warmup_ratio=warmup_ratio,
        logging_steps=10,
        eval_strategy="epoch",
        save_strategy="no",
        bf16=USE_BF16,
        fp16=not USE_BF16,
        optim=optim,
        seed=SEED,
        report_to="none",
        run_name=run_name,
    )
    if gradient_checkpointing:
        common_targs["gradient_checkpointing"] = True

    trainer_kwargs = {}
    if accepts_directly or SFTConfig is None:
        args_obj = TrainingArguments(**common_targs)
        trainer_kwargs.update(dataset_text_field="text", max_seq_length=MAX_SEQ_LENGTH)
    else:
        args_obj = SFTConfig(dataset_text_field="text", max_seq_length=MAX_SEQ_LENGTH,
                             **common_targs)

    trainer_kwargs[_tok_kwarg(SFTTrainer)] = tok
    return SFTTrainer(model=model, train_dataset=train_ds, eval_dataset=val_ds,
                      args=args_obj, **trainer_kwargs)

def _build_dpo_trainer(model, tok, train_ds, val_ds, output_dir, *, beta,
                       num_train_epochs, per_device_train_batch_size,
                       gradient_accumulation_steps, learning_rate, run_name):
    from trl import DPOConfig, DPOTrainer
    args_obj = DPOConfig(
        output_dir=output_dir,
        beta=beta,
        num_train_epochs=num_train_epochs,
        per_device_train_batch_size=per_device_train_batch_size,
        gradient_accumulation_steps=gradient_accumulation_steps,
        learning_rate=learning_rate,
        lr_scheduler_type="cosine",
        warmup_ratio=0.1,
        logging_steps=10,
        eval_strategy="epoch",
        save_strategy="no",
        bf16=USE_BF16,
        fp16=not USE_BF16,
        optim="paged_adamw_8bit",
        max_length=MAX_SEQ_LENGTH,
        max_prompt_length=MAX_SEQ_LENGTH - 512,
        seed=SEED,
        report_to="none",
        run_name=run_name,
    )
    kwargs = {_tok_kwarg(DPOTrainer): tok}
    return DPOTrainer(model=model, ref_model=None, train_dataset=train_ds,
                      eval_dataset=val_ds, args=args_obj, **kwargs)

print("Adaptive trainer-construction helpers ready.")


## Step 5 — QLoRA fine-tuning (Mistral-7B-Instruct-v0.3, Unsloth, 4-bit)

**Hyperparameters**: identical to `finetune_qlora.py`'s own defaults
(`rank=16, alpha=32, lr=2e-4, batch_size=2, grad_accum=8, epochs=3`) —
these are the script's real, non-reduced recipe (documented VRAM need:
~7 GB), not the smaller `batch_size=1, max_samples=256` used for the local
4 GB-GPU smoke test. On a T4 there's no need to shrink batch size or cap
the dataset — this is the first run with enough VRAM to use the intended
settings, on the full 606-example training set.

`bf16=USE_BF16` (computed in Step 0) replaces the original script's
hardcoded `bf16=True`, since T4 doesn't support bf16 natively.


In [ ]:
print(f"Loading {BASE_MODEL_UNSLOTH_4BIT} (4-bit, Unsloth)...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL_UNSLOTH_4BIT,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.0,          # 0 enables Unsloth's fast path
    target_modules=LORA_TARGETS,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

from datasets import Dataset

to_text = lambda rows: Dataset.from_dict({
    "text": [format_example(r["input"], r["output"], tokenizer.eos_token) for r in rows]
})
qlora_train_ds, qlora_val_ds = to_text(train_rows), to_text(val_rows)
print(f"Train: {len(qlora_train_ds)} | Val: {len(qlora_val_ds)}")

QLORA_OUTPUT_DIR = os.path.join(MODELS_DIR, "qlora_7b")
qlora_tracker = RunTracker("qlora_7b", QLORA_OUTPUT_DIR)
qlora_tracker.start()

qlora_trainer = _build_sft_trainer(
    model, tokenizer, qlora_train_ds, qlora_val_ds, QLORA_OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,   # effective batch = 16
    learning_rate=2e-4,
    optim="adamw_8bit",
    run_name="qlora-mistral-7b",
)
qlora_trainer.train()


In [ ]:
# ── Real measured results for QLoRA (printed here, not estimated) ─────────
qlora_train_loss, qlora_eval_loss = get_final_losses(qlora_trainer)
qlora_tracker.stop(final_loss=qlora_train_loss, eval_loss=qlora_eval_loss, extra={
    "base_model": BASE_MODEL_UNSLOTH_4BIT,
    "method": "QLoRA (4-bit NF4 base, Unsloth)",
    "rank": 16, "alpha": 32, "lr": 2e-4, "epochs": 3,
    "n_train": len(qlora_train_ds),
})

model.save_pretrained(QLORA_OUTPUT_DIR)
tokenizer.save_pretrained(QLORA_OUTPUT_DIR)
qlora_tracker.save()
backup_to_drive(QLORA_OUTPUT_DIR, "qlora_7b")

free_gpu(model, tokenizer, qlora_trainer)


## Step 6 — DoRA fine-tuning (PEFT, `use_dora=True`) — apples-to-apples vs QLoRA

Same base model, same data, same rank/alpha/lr/batch/grad-accum/epochs as
the QLoRA cell above — the **only** difference is `use_dora=True` in the
`LoraConfig`, matching `finetune_dora.py`'s design intent exactly ("clean
side-by-side... same data, same prompt format, same rank"). DoRA uses plain
PEFT + bitsandbytes (not Unsloth) per the original script, since Unsloth's
fused kernels don't cover DoRA's magnitude/direction decomposition path.

**If you're worried about time or memory at this point**: DoRA is
independent of the DPO stage below (DPO only needs the QLoRA adapter you
already saved). You can skip straight to Step 8 (DPO) and come back to DoRA
later if this cell is at risk of not finishing.


In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print(f"Loading {BASE_MODEL_HF} in 4-bit NF4 via bitsandbytes...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if USE_BF16 else torch.float16,
)
dora_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_HF, quantization_config=bnb_config, device_map="auto")
dora_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_HF)
if dora_tokenizer.pad_token is None:
    dora_tokenizer.pad_token = dora_tokenizer.eos_token

dora_model = prepare_model_for_kbit_training(dora_model)
dora_model = get_peft_model(dora_model, LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=LORA_TARGETS, bias="none", task_type="CAUSAL_LM",
    use_dora=True,   # <- the only real difference vs the QLoRA cell above
))
dora_model.print_trainable_parameters()

from datasets import Dataset
dora_train_ds = to_text(train_rows) if 'to_text' in dir() else Dataset.from_dict({
    "text": [format_example(r["input"], r["output"], dora_tokenizer.eos_token) for r in train_rows]})
dora_val_ds = Dataset.from_dict({
    "text": [format_example(r["input"], r["output"], dora_tokenizer.eos_token) for r in val_rows]})
print(f"Train: {len(dora_train_ds)} | Val: {len(dora_val_ds)}")

DORA_OUTPUT_DIR = os.path.join(MODELS_DIR, "dora_7b")
dora_tracker = RunTracker("dora_7b", DORA_OUTPUT_DIR)
dora_tracker.start()

dora_trainer = _build_sft_trainer(
    dora_model, dora_tokenizer, dora_train_ds, dora_val_ds, DORA_OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,   # SAME effective batch as QLoRA (16)
    learning_rate=2e-4,
    optim="paged_adamw_8bit",
    run_name="dora-mistral-7b",
    gradient_checkpointing=True,
)
dora_trainer.train()


In [ ]:
# ── Real measured results for DoRA (printed here, not estimated) ──────────
dora_train_loss, dora_eval_loss = get_final_losses(dora_trainer)
dora_tracker.stop(final_loss=dora_train_loss, eval_loss=dora_eval_loss, extra={
    "base_model": BASE_MODEL_HF,
    "method": "DoRA (weight-decomposed LoRA, PEFT, 4-bit base)",
    "rank": 16, "alpha": 32, "lr": 2e-4, "epochs": 3,
    "n_train": len(dora_train_ds),
})

dora_model.save_pretrained(DORA_OUTPUT_DIR)
dora_tokenizer.save_pretrained(DORA_OUTPUT_DIR)
dora_tracker.save()
backup_to_drive(DORA_OUTPUT_DIR, "dora_7b")

free_gpu(dora_model, dora_tokenizer, dora_trainer)


## Step 7 — DPO on top of the QLoRA adapter

Loads the QLoRA adapter saved in Step 5 and continues training with
preference pairs (chosen vs rejected explanations for the same SHAP input),
matching `finetune_dpo.py`'s Unsloth branch exactly. On Colab (Linux) the
standard Unsloth adapter-loading path is used directly — the local script's
Windows-only state-dict workaround (for a `pyarrow`/`bitsandbytes` access
violation specific to Windows DLL loading) does not apply here and is not
needed.


In [ ]:
from unsloth import PatchDPOTrainer
PatchDPOTrainer()

print(f"Loading QLoRA adapter from {QLORA_OUTPUT_DIR} (Unsloth)...")
dpo_model, dpo_tokenizer = FastLanguageModel.from_pretrained(
    model_name=QLORA_OUTPUT_DIR,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)
# Defensive guard (Mistral has a real BOS token so this is a no-op here;
# kept because some trl versions' DPOTrainer prepend bos_token_id
# unconditionally and would crash on any tokenizer where it's None).
if dpo_tokenizer.bos_token_id is None:
    dpo_tokenizer.bos_token = dpo_tokenizer.eos_token

from datasets import Dataset
to_pref_ds = lambda rows: Dataset.from_dict({
    "prompt": [format_prompt(r["prompt"]) for r in rows],
    "chosen": [" " + r["chosen"] for r in rows],
    "rejected": [" " + r["rejected"] for r in rows],
})
dpo_train_ds, dpo_val_ds = to_pref_ds(pref_train_rows), to_pref_ds(pref_val_rows)
print(f"Preference pairs — train: {len(dpo_train_ds)} | val: {len(dpo_val_ds)}")

DPO_OUTPUT_DIR = os.path.join(MODELS_DIR, "dpo_7b")
dpo_tracker = RunTracker("dpo_7b", DPO_OUTPUT_DIR)
dpo_tracker.start()

dpo_trainer = _build_dpo_trainer(
    dpo_model, dpo_tokenizer, dpo_train_ds, dpo_val_ds, DPO_OUTPUT_DIR,
    beta=0.1,
    num_train_epochs=1,              # DPO overfits fast — 1 epoch is correct here
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-6,              # much lower than SFT — DPO gradients are sharp
    run_name="dpo-on-qlora-mistral-7b",
)
dpo_trainer.train()


In [ ]:
# ── Real measured results for DPO (printed here, not estimated) ───────────
dpo_train_loss, dpo_eval_loss = get_final_losses(dpo_trainer)
dpo_tracker.stop(final_loss=dpo_train_loss, eval_loss=dpo_eval_loss, extra={
    "base_model": QLORA_OUTPUT_DIR,
    "method": "DPO (preference pairs, TRL, Unsloth, on top of QLoRA)",
    "beta": 0.1, "lr": 5e-6, "epochs": 1,
    "started_from_sft_adapter": True,
    "n_train_pairs": len(dpo_train_ds),
})

dpo_model.save_pretrained(DPO_OUTPUT_DIR)
dpo_tokenizer.save_pretrained(DPO_OUTPUT_DIR)
dpo_tracker.save()
backup_to_drive(DPO_OUTPUT_DIR, "dpo_7b")

free_gpu(dpo_model, dpo_tokenizer, dpo_trainer)


## Step 8 — Recap: real measured stats for all three 7B runs

In [ ]:
print(f"{'Run':<10} {'Train time':<14} {'Peak VRAM (GB)':<16} {'Final train loss':<18} {'Final eval loss'}")
for name, stats_path in [
    ("QLoRA", os.path.join(QLORA_OUTPUT_DIR, "training_stats.json")),
    ("DoRA",  os.path.join(DORA_OUTPUT_DIR, "training_stats.json")),
    ("DPO",   os.path.join(DPO_OUTPUT_DIR, "training_stats.json")),
]:
    if os.path.exists(stats_path):
        s = json.load(open(stats_path))
        print(f"{name:<10} {s.get('train_time_human','-'):<14} "
              f"{s.get('peak_gpu_memory_gb','-'):<16} "
              f"{s.get('final_train_loss','-'):<18} "
              f"{s.get('final_eval_loss','-')}")
    else:
        print(f"{name:<10} (not run — stats file not found at {stats_path})")

print("\nNote: DPO's loss is a preference loss, not directly comparable to "
      "QLoRA/DoRA's SFT cross-entropy loss — compare DPO via the judge "
      "scores in the benchmark below instead.")


## Step 9 — Build a real `llama-quantize` (fixes the f16-only gap from the last run)

The earlier local export had no C++ toolchain available and shipped f16
GGUF only (~14 GB for a 7B). Here we actually install `cmake` +
`build-essential`, clone `llama.cpp`, and compile — building **only** the
`llama-quantize` target (not the full server/CLI suite) and with CUDA
disabled, since quantization itself is a CPU-bound tensor-reformatting
operation with negligible GPU benefit — this keeps the build to a few
minutes instead of ~15-20 for a full CUDA build.


In [ ]:
%%capture cmake_log
!apt-get -qq update
!apt-get -qq install -y cmake build-essential
!git clone --depth 1 https://github.com/ggerganov/llama.cpp /content/llama.cpp
!pip install -q -r /content/llama.cpp/requirements.txt


In [ ]:
print("\n".join(cmake_log.stdout.splitlines()[-30:]))


In [ ]:
import subprocess

LLAMA_CPP_DIR = "/content/llama.cpp"

build_cfg = subprocess.run(
    ["cmake", "-B", "build", "-DGGML_CUDA=OFF", "-DCMAKE_BUILD_TYPE=Release"],
    cwd=LLAMA_CPP_DIR, capture_output=True, text=True)
print(build_cfg.stdout[-2000:])
print(build_cfg.stderr[-2000:])
assert build_cfg.returncode == 0, "cmake configure failed — see output above"

build_run = subprocess.run(
    ["cmake", "--build", "build", "--config", "Release", "-j",
     str(os.cpu_count()), "--target", "llama-quantize"],
    cwd=LLAMA_CPP_DIR, capture_output=True, text=True)
print(build_run.stdout[-2000:])
print(build_run.stderr[-2000:])
assert build_run.returncode == 0, "cmake build failed — see output above"

QUANTIZE_BIN = None
for candidate in ("build/bin/llama-quantize", "build/llama-quantize"):
    p = os.path.join(LLAMA_CPP_DIR, candidate)
    if os.path.exists(p):
        QUANTIZE_BIN = p
        break
assert QUANTIZE_BIN is not None, "llama-quantize binary not found after build — check build_run output above"
print(f"\nBuilt real llama-quantize binary at: {QUANTIZE_BIN}")


## Step 10 — Merge the best adapter, convert to GGUF, quantize to q4_k_m

**Ordering note**: the 4-way LLM-judged benchmark (which would tell us
definitively which adapter is "best") runs in Step 12, *after* this export
step, per the order requested. Since the real benchmark result isn't known
yet at this point, `EXPORT_TECHNIQUE` defaults to `"qlora_7b"` (justified by
being the directly-comparable-loss winner candidate between QLoRA/DoRA at
this point, and the base DPO was built on). Everything below is written as
a reusable function — after Step 12 prints the real benchmark table, a
follow-up cell (Step 13) lets you re-run this with a different technique in
one line if the benchmark shows a different adapter actually won.


In [ ]:
EXPORT_DIR = "/content/finetuning/models/export"
os.makedirs(EXPORT_DIR, exist_ok=True)
QUANT_TYPE = "Q4_K_M"   # real quantization this time, not f16

def merge_and_quantize(technique_dir: str, technique_name: str, quant: str = QUANT_TYPE):
    """Merge (W' = W + BA) -> GGUF f16 -> quantize. Returns the final .gguf path."""
    merged_dir = os.path.join(EXPORT_DIR, f"{technique_name}_merged")

    print(f"[merge] Merging {technique_name} adapter into fp16 via Unsloth...")
    merge_model, merge_tokenizer = FastLanguageModel.from_pretrained(
        model_name=technique_dir, max_seq_length=MAX_SEQ_LENGTH, load_in_4bit=True)
    merge_model.save_pretrained_merged(merged_dir, merge_tokenizer, save_method="merged_16bit")
    free_gpu(merge_model, merge_tokenizer)
    print(f"[merge] Merged fp16 model saved to {merged_dir}")

    f16_gguf = os.path.join(EXPORT_DIR, f"{technique_name}_f16.gguf")
    final_gguf = os.path.join(EXPORT_DIR, f"datapilot-explainer_{technique_name}_{quant.lower()}.gguf")

    print(f"[gguf] Converting to f16 GGUF...")
    conv = subprocess.run(
        ["python", os.path.join(LLAMA_CPP_DIR, "convert_hf_to_gguf.py"),
         merged_dir, "--outfile", f16_gguf, "--outtype", "f16"],
        capture_output=True, text=True)
    print(conv.stdout[-1500:]); print(conv.stderr[-1500:])
    assert conv.returncode == 0, "GGUF conversion failed — see output above"
    print(f"[gguf] f16 GGUF: {f16_gguf} ({os.path.getsize(f16_gguf)/1024**3:.2f} GB)")

    print(f"[gguf] Quantizing to {quant} with the REAL compiled llama-quantize...")
    quant_run = subprocess.run([QUANTIZE_BIN, f16_gguf, final_gguf, quant],
                               capture_output=True, text=True)
    print(quant_run.stdout[-1500:]); print(quant_run.stderr[-1500:])
    assert quant_run.returncode == 0, "Quantization failed — see output above"
    os.remove(f16_gguf)  # large intermediate, delete once quantized

    final_size_gb = os.path.getsize(final_gguf) / 1024**3
    print(f"[gguf] DONE: {final_gguf} ({final_size_gb:.2f} GB, {quant})")
    return final_gguf

EXPORT_TECHNIQUE = "qlora_7b"   # default pick — see note above; changeable after Step 12
TECHNIQUE_DIRS = {"qlora_7b": QLORA_OUTPUT_DIR, "dora_7b": DORA_OUTPUT_DIR, "dpo_7b": DPO_OUTPUT_DIR}

exported_gguf_path = merge_and_quantize(TECHNIQUE_DIRS[EXPORT_TECHNIQUE], EXPORT_TECHNIQUE)


## Step 11 — Generate the Ollama Modelfile and package for download

Same template as `export_ollama.py::write_modelfile()` — the Mistral
`[INST]` chat template plus the Explainer system prompt baked in, so the
tuned behavior survives inside Ollama with zero extra configuration.


In [ ]:
def write_modelfile(gguf_path: str, out_path: str) -> str:
    content = f'''# DataPilot Explainer -- fine-tuned Mistral-7B-Instruct
# Turns SHAP/LIME attribution output into plain-English explanations.
FROM {os.path.basename(gguf_path)}

TEMPLATE """[INST] {{{{ if .System }}}}{{{{ .System }}}}

{{{{ end }}}}{{{{ .Prompt }}}} [/INST]"""

SYSTEM """{SYSTEM_INSTRUCTION}"""

PARAMETER temperature 0.3
PARAMETER num_ctx 2048
PARAMETER stop "[INST]"
PARAMETER stop "[/INST]"
'''
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(content)
    return out_path

modelfile_path = os.path.join(EXPORT_DIR, "Modelfile")
write_modelfile(exported_gguf_path, modelfile_path)
print(f"Modelfile written to {modelfile_path}:\n")
print(open(modelfile_path).read())


In [ ]:
import shutil, zipfile

# Package GGUF + Modelfile into a single downloadable archive. Note: an
# already-quantized GGUF is compressed binary data, so zipping it saves
# negligible extra space — this is purely for "one file to grab" convenience.
package_path = f"/content/{EXPORT_TECHNIQUE}_datapilot_explainer.zip"
with zipfile.ZipFile(package_path, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(exported_gguf_path, os.path.basename(exported_gguf_path))
    zf.write(modelfile_path, "Modelfile")

package_size_gb = os.path.getsize(package_path) / 1024**3
print(f"Package ready: {package_path} ({package_size_gb:.2f} GB)")

# ── Primary: copy to Drive (more reliable than browser download for
# multi-GB files) ────────────────────────────────────────────────────────
if DRIVE_BACKUP_DIR is not None:
    drive_dest = os.path.join(DRIVE_BACKUP_DIR, os.path.basename(package_path))
    shutil.copy(package_path, drive_dest)
    print(f"Copied to Drive: {drive_dest}")
else:
    print("Drive not mounted — use the direct-download cell below instead.")


In [ ]:
# ── Alternative: direct browser download ───────────────────────────────────
# Set to True to trigger it — can be slow/unreliable for multi-GB files
# depending on your connection; prefer the Drive copy above when possible.
RUN_DIRECT_DOWNLOAD = False

if RUN_DIRECT_DOWNLOAD:
    from google.colab import files
    files.download(package_path)
else:
    print("Skipped (RUN_DIRECT_DOWNLOAD is False). Flip to True to download "
          f"{package_path} directly via the browser.")


## Step 12 — Four-way benchmark: base Mistral-7B vs 0.5B smoke vs 7B QLoRA vs 7B DoRA

Same methodology as `finetuning/benchmark.py`: generate on the same 12
held-out test examples (the first 12 rows of the exact `test.jsonl` loaded
in Step 3 — same file, same order, so this is genuinely the same 12
examples used in the earlier local smoke benchmark), then score each
generation 1-5 on clarity/accuracy with an LLM judge, plus the deterministic
jargon-leak check.

DPO is trained and measured above but is **not** one of the four requested
here (base / 0.5B smoke / 7B QLoRA / 7B DoRA) — add it in one line by
appending `("dpo_7b", DPO_OUTPUT_DIR, "unsloth", None)` to `CANDIDATES`
below if you want it in the table too.

**Generation runs first** (needs GPU, one adapter in VRAM at a time, freed
between each). **Ollama + the judge model start only after all generation
is done and VRAM is released** — running both at once risks memory
contention between the HF/Unsloth generation stack and Ollama's own 7B
judge model.


### Step 12a — Upload the existing 0.5B smoke adapter

In [ ]:
# The 0.5B smoke adapter (finetuning/models/qlora on the local machine) is
# NOT in git (checkpoint binaries are intentionally untracked) — upload it
# here. On your local machine:
#   cd finetuning/models && zip -r qlora_0.5b_smoke.zip qlora
# then select that zip below.
from google.colab import files

print("Select your zipped 0.5B smoke adapter (finetuning/models/qlora)...")
uploaded = files.upload()
smoke_zip_name = next(iter(uploaded))

SMOKE_ADAPTER_DIR = "/content/finetuning/models/qlora_0.5b_smoke"
os.makedirs(SMOKE_ADAPTER_DIR, exist_ok=True)
with zipfile.ZipFile(smoke_zip_name) as zf:
    zf.extractall(SMOKE_ADAPTER_DIR)

# Handle the zip having wrapped everything in a top-level "qlora/" folder
inner = os.path.join(SMOKE_ADAPTER_DIR, "qlora")
if os.path.exists(os.path.join(inner, "adapter_config.json")):
    for f in os.listdir(inner):
        shutil.move(os.path.join(inner, f), os.path.join(SMOKE_ADAPTER_DIR, f))
    os.rmdir(inner)

assert os.path.exists(os.path.join(SMOKE_ADAPTER_DIR, "adapter_config.json")), \
    "adapter_config.json not found after extraction — check the zip's structure"
smoke_base = json.load(open(os.path.join(SMOKE_ADAPTER_DIR, "adapter_config.json")))["base_model_name_or_path"]
print(f"0.5B smoke adapter loaded. Its base model: {smoke_base}")


### Step 12b — Generate with all four candidates (same 12 test examples)

In [ ]:
BENCHMARK_TEST_ROWS = test_rows[:12]
print(f"Benchmarking on {len(BENCHMARK_TEST_ROWS)} held-out examples "
      f"(first 12 rows of test.jsonl loaded in Step 3).")

def generate_unsloth(model_name_or_path, rows, max_new_tokens=200):
    gen_model, gen_tok = FastLanguageModel.from_pretrained(
        model_name=model_name_or_path, max_seq_length=MAX_SEQ_LENGTH, load_in_4bit=True)
    FastLanguageModel.for_inference(gen_model)
    outputs = []
    for row in rows:
        prompt = format_prompt(row["input"])
        inputs = gen_tok(prompt, return_tensors="pt").to(gen_model.device)
        with torch.no_grad():
            out = gen_model.generate(**inputs, max_new_tokens=max_new_tokens,
                                     temperature=0.3, do_sample=True,
                                     pad_token_id=gen_tok.eos_token_id)
        text = gen_tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        outputs.append(text)
    free_gpu(gen_model, gen_tok)
    return outputs

def generate_transformers(base_model_id, adapter_path, rows, max_new_tokens=200):
    from peft import PeftModel
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_use_double_quant=True,
                             bnb_4bit_compute_dtype=torch.bfloat16 if USE_BF16 else torch.float16)
    gen_model = AutoModelForCausalLM.from_pretrained(base_model_id, quantization_config=bnb, device_map="auto")
    gen_tok = AutoTokenizer.from_pretrained(base_model_id)
    if gen_tok.pad_token is None:
        gen_tok.pad_token = gen_tok.eos_token
    if adapter_path:
        gen_model = PeftModel.from_pretrained(gen_model, adapter_path)
    gen_model.eval()
    outputs = []
    for row in rows:
        prompt = format_prompt(row["input"])
        inputs = gen_tok(prompt, return_tensors="pt").to(gen_model.device)
        with torch.no_grad():
            out = gen_model.generate(**inputs, max_new_tokens=max_new_tokens,
                                     temperature=0.3, do_sample=True,
                                     pad_token_id=gen_tok.eos_token_id)
        text = gen_tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        outputs.append(text)
    free_gpu(gen_model, gen_tok)
    return outputs

# (name, generation-fn call) — base + 3 adapters = 4-way comparison
CANDIDATES = [
    ("base_mistral_7b_untuned", lambda: generate_unsloth(BASE_MODEL_UNSLOTH_4BIT, BENCHMARK_TEST_ROWS)),
    ("qlora_0.5b_smoke", lambda: generate_transformers(smoke_base, SMOKE_ADAPTER_DIR, BENCHMARK_TEST_ROWS)),
    ("qlora_7b", lambda: generate_unsloth(QLORA_OUTPUT_DIR, BENCHMARK_TEST_ROWS)),
    ("dora_7b", lambda: generate_transformers(BASE_MODEL_HF, DORA_OUTPUT_DIR, BENCHMARK_TEST_ROWS)),
]

RESULTS_DIR = "/content/finetuning/results"
os.makedirs(RESULTS_DIR, exist_ok=True)
generations = {}

for name, gen_fn in CANDIDATES:
    print(f"\n=== Generating: {name} ===")
    outputs = gen_fn()
    generations[name] = outputs
    gen_path = os.path.join(RESULTS_DIR, f"generations_{name}.jsonl")
    with open(gen_path, "w", encoding="utf-8") as f:
        for row, out in zip(BENCHMARK_TEST_ROWS, outputs):
            f.write(json.dumps({"input": row["input"], "reference": row["output"], "generated": out}) + "\n")
    for i, out in enumerate(outputs[:3]):
        print(f"  [{i}] {out[:150]}...")
    print(f"  ({len(outputs)} generations saved to {gen_path})")

print("\nAll generation done. GPU is free — safe to start Ollama next.")


### Step 12c — Install Ollama, pull the judge model, score everything

In [ ]:
%%capture ollama_install_log
!curl -fsSL https://ollama.com/install.sh | sh


In [ ]:
print("\n".join(ollama_install_log.stdout.splitlines()[-20:]))

import subprocess, time
ollama_proc = subprocess.Popen(["ollama", "serve"],
                               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
pull = subprocess.run(["ollama", "pull", "mistral:7b-instruct"], capture_output=True, text=True)
print(pull.stdout[-1000:]); print(pull.stderr[-1000:])
assert pull.returncode == 0, "ollama pull failed — check output above"
print("\nJudge model ready: mistral:7b-instruct")


In [ ]:
import requests, re, statistics

OLLAMA_URL = "http://localhost:11434"
JUDGE_MODEL = "mistral:7b-instruct"

JUDGE_PROMPT = """You are grading an AI-written explanation of a machine-learning prediction.

GROUND TRUTH (the model's actual feature attributions):
{shap_input}

EXPLANATION TO GRADE:
{explanation}

Score two criteria from 1 to 5 (integers only):

CLARITY -- could a non-technical manager understand it?
  5 = plain English, short, no jargon, well structured
  3 = understandable but clunky or slightly jargony
  1 = jargon dump or incoherent

ACCURACY -- does it match the ground truth attributions?
  5 = names the truly dominant factors with correct directions
  1 = wrong factors or wrong directions

Note: short and clear beats long. Do NOT reward length.

Answer with ONLY this JSON: {{"clarity": <int>, "accuracy": <int>}}"""

JARGON_RE = re.compile(r"\b(shap|lime|attribution|logit|coefficient)\b", re.I)

def judge_one(shap_input, explanation):
    try:
        r = requests.post(f"{OLLAMA_URL}/api/generate", json={
            "model": JUDGE_MODEL,
            "prompt": JUDGE_PROMPT.format(shap_input=shap_input, explanation=explanation),
            "format": "json", "stream": False,
            "options": {"temperature": 0.0, "num_predict": 60}}, timeout=90)
        r.raise_for_status()
        parsed = json.loads(r.json()["response"])
        c, a = int(parsed["clarity"]), int(parsed["accuracy"])
        if 1 <= c <= 5 and 1 <= a <= 5:
            return {"clarity": c, "accuracy": a}
    except Exception:
        pass
    return None

def judge_all(rows, outputs):
    clarities, accuracies = [], []
    for row, out in zip(rows, outputs):
        score = judge_one(row["input"], out)
        if score:
            clarities.append(score["clarity"]); accuracies.append(score["accuracy"])
    if not clarities:
        return {"clarity": float("nan"), "accuracy": float("nan"), "overall": float("nan"), "n_scored": 0}
    return {"clarity": round(statistics.mean(clarities), 2),
            "accuracy": round(statistics.mean(accuracies), 2),
            "overall": round((statistics.mean(clarities) + statistics.mean(accuracies)) / 2, 2),
            "n_scored": len(clarities)}

def rule_metrics(outputs):
    n = max(len(outputs), 1)
    return {"jargon_leak_pct": round(100 * sum(bool(JARGON_RE.search(o)) for o in outputs) / n, 1),
            "avg_length_chars": round(sum(len(o) for o in outputs) / n)}

def load_training_stats(technique_dir):
    p = os.path.join(technique_dir, "training_stats.json")
    return json.load(open(p)) if os.path.exists(p) else {}

STATS_DIRS = {"base_mistral_7b_untuned": None, "qlora_0.5b_smoke": SMOKE_ADAPTER_DIR,
              "qlora_7b": QLORA_OUTPUT_DIR, "dora_7b": DORA_OUTPUT_DIR}

results = []
for name, outputs in generations.items():
    print(f"\nJudging: {name} ...")
    judge = judge_all(BENCHMARK_TEST_ROWS, outputs)
    rules = rule_metrics(outputs)
    stats = load_training_stats(STATS_DIRS[name]) if STATS_DIRS[name] else {}
    results.append({"name": name,
                    "train_time": stats.get("train_time_human", "-"),
                    "peak_vram": stats.get("peak_gpu_memory_gb", "-"),
                    "train_loss": stats.get("final_train_loss", "-"),
                    "eval_loss": stats.get("final_eval_loss", "-"),
                    **judge, **rules})
    print(f"  clarity={judge['clarity']} accuracy={judge['accuracy']} "
          f"overall={judge['overall']} jargon_leak%={rules['jargon_leak_pct']} "
          f"(judged {judge['n_scored']}/{len(BENCHMARK_TEST_ROWS)})")


In [ ]:
# ── Final four-way markdown table (same format as finetuning/benchmark.py) ─
results_sorted = sorted(results, key=lambda r: (r["overall"] if r["overall"] == r["overall"] else 0), reverse=True)

lines = ["# Four-Way Benchmark Results — Mistral-7B (Colab)", "",
         f"Test set: {len(BENCHMARK_TEST_ROWS)} held-out examples (same file/order as the local smoke "
         f"benchmark's test.jsonl) | Judge: {JUDGE_MODEL} via Ollama (clarity + accuracy, 1-5)", "",
         "| Candidate | Train time | Peak VRAM (GB) | Train loss | Eval loss | "
         "Clarity (1-5) | Accuracy (1-5) | Overall | Jargon leak % | Judged n |",
         "|---|---|---|---|---|---|---|---|---|---|"]
for r in results_sorted:
    lines.append(f"| {r['name']} | {r['train_time']} | {r['peak_vram']} | {r['train_loss']} | "
                f"{r['eval_loss']} | {r['clarity']} | {r['accuracy']} | **{r['overall']}** | "
                f"{r['jargon_leak_pct']} | {r['n_scored']} |")

md_table = "\n".join(lines)
print(md_table)

results_path = os.path.join(RESULTS_DIR, "benchmark_4way_results.md")
with open(results_path, "w", encoding="utf-8") as f:
    f.write(md_table)
print(f"\nSaved to {results_path}")
if DRIVE_BACKUP_DIR is not None:
    shutil.copy(results_path, os.path.join(DRIVE_BACKUP_DIR, "benchmark_4way_results.md"))
    for name in generations:
        shutil.copy(os.path.join(RESULTS_DIR, f"generations_{name}.jsonl"),
                    os.path.join(DRIVE_BACKUP_DIR, f"generations_{name}.jsonl"))
    print("Results + raw generations backed up to Drive.")


## Step 13 — Optional: re-export if the benchmark crowned a different winner

Step 10 defaulted to exporting `qlora_7b` before real benchmark numbers
existed. If the table above shows `dora_7b` (or `dpo_7b`, not in the 4-way
table but trained in Step 7) with a meaningfully higher **Overall** judge
score, re-export that one instead — one line, reuses the same
`merge_and_quantize()` function and the real compiled `llama-quantize`.


In [ ]:
RE_EXPORT_TECHNIQUE = None   # e.g. "dora_7b" or "dpo_7b" — set and run if the benchmark disagreed

if RE_EXPORT_TECHNIQUE:
    TECHNIQUE_DIRS["dpo_7b"] = DPO_OUTPUT_DIR
    exported_gguf_path = merge_and_quantize(TECHNIQUE_DIRS[RE_EXPORT_TECHNIQUE], RE_EXPORT_TECHNIQUE)
    modelfile_path = os.path.join(EXPORT_DIR, "Modelfile")
    write_modelfile(exported_gguf_path, modelfile_path)
    package_path = f"/content/{RE_EXPORT_TECHNIQUE}_datapilot_explainer.zip"
    with zipfile.ZipFile(package_path, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(exported_gguf_path, os.path.basename(exported_gguf_path))
        zf.write(modelfile_path, "Modelfile")
    print(f"Re-exported: {package_path} ({os.path.getsize(package_path)/1024**3:.2f} GB)")
    if DRIVE_BACKUP_DIR is not None:
        shutil.copy(package_path, os.path.join(DRIVE_BACKUP_DIR, os.path.basename(package_path)))
        print("Copied to Drive.")
else:
    print("RE_EXPORT_TECHNIQUE is None — keeping the Step 10 export "
          f"({EXPORT_TECHNIQUE}). Set it above and re-run this cell to change.")


## Done — what you have now

- Three real 7B adapters (`qlora_7b`, `dora_7b`, `dpo_7b`) with measured
  time/VRAM/loss, backed up to Drive after each stage.
- A real `Q4_K_M`-quantized GGUF (~4.1-4.4 GB, not the previous f16-only
  export) + Modelfile, packaged as a single zip.
- A real four-way judged benchmark table comparing base Mistral-7B, the
  0.5B smoke adapter, the 7B QLoRA adapter, and the 7B DoRA adapter on the
  same 12 held-out examples used before.

**Explicitly not run in this notebook (already decided, out of scope):**
ORPO and GaLore.

**Next step on your local machine**: unzip the downloaded package, then
```bash
ollama create datapilot-explainer -f Modelfile
# .env:  OLLAMA_MODEL=datapilot-explainer
```
